![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 10 -- Lab 1: Transfer Learning -- Cats vs Dogs

You have a small dataset of cat and dog images. Training a deep CNN from scratch would be slow and prone to overfitting. Instead, you will leverage a model that has already learned rich visual features from ImageNet and adapt it to your binary classification task.

| Part | Goal |
|---|---|
| Setup & Data | Load Cats & Dogs dataset, build DataLoaders |
| Baseline | Train EfficientNet-B0 from random weights (no pretraining) |
| Strategy 1 | Feature extraction -- freeze backbone, train only the head |
| Strategy 2 | Fine-tuning -- unfreeze last layers with differential LR |
| Comparison | Compare all three approaches side by side |

**Key takeaway:** Pretrained models + transfer learning let you achieve strong results with limited data and compute.

In [ ]:
!pip install kagglehub -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1 -- Setup and Data (GIVEN)

We use a small Cats & Dogs dataset from Kaggle. The dataset comes with train/test CSV files that list image paths and labels.

In [ ]:
# --- GIVEN: Download the dataset ---
path = kagglehub.dataset_download("marquis03/cats-and-dogs")
print(f'Dataset downloaded to: {path}')

In [ ]:
# --- GIVEN: Custom Dataset class ---
class CatsDogsDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = root_dir
        self.transform = transform
        csv_file = os.path.join(root_dir, f'{split}.csv')
        self.data = pd.read_csv(csv_file)
        self.image_paths = self.data['image:FILE'].values
        self.labels = self.data['category'].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_paths[idx])
        image = Image.open(img_path).convert('RGB')
        label = float(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# --- GIVEN: Transforms and DataLoaders ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = CatsDogsDataset(path, split='train', transform=train_transform)
valid_dataset = CatsDogsDataset(path, split='test',  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(valid_dataset)}')

In [ ]:
# --- GIVEN: Visualize sample images ---
class_names = ['Cat', 'Dog']

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    ax.set_title(class_names[int(label)], fontsize=11)
    ax.axis('off')

fig.suptitle('Cats & Dogs Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 1.5 -- Training and Validation Helpers (GIVEN)

These helper functions run one epoch of training and one pass of validation. You will reuse them for every strategy.

In [ ]:
# --- GIVEN: Training helper ---
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(dataloader, leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = torch.sigmoid(outputs) > 0.5
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total


# --- GIVEN: Validation helper ---
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predictions = torch.sigmoid(outputs) > 0.5
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total


# --- GIVEN: Full training loop ---
def run_training(model, train_loader, valid_loader, criterion, optimizer, device, num_epochs):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, valid_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch+1}/{num_epochs}: '
              f'Train Loss={train_loss:.4f}, Train Acc={train_acc:.1f}%, '
              f'Val Loss={val_loss:.4f}, Val Acc={val_acc:.1f}%')

    return history


# --- GIVEN: Plotting helper ---
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    epochs = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs, history['train_loss'], 'o-', label='Train')
    ax1.plot(epochs, history['val_loss'], 's-', label='Validation')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} -- Loss'); ax1.legend()

    ax2.plot(epochs, history['train_acc'], 'o-', label='Train')
    ax2.plot(epochs, history['val_acc'], 's-', label='Validation')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{title} -- Accuracy'); ax2.legend()

    plt.tight_layout()
    plt.show()

---
# Part 2 -- Baseline: No Pretraining

First, let's see how well EfficientNet-B0 performs when trained **from scratch** (random initialization, no pretrained weights). This gives us a baseline to compare transfer learning against.

## Task 1: Load EfficientNet-B0 Without Pretrained Weights

Fill in the `None` values:
- Load EfficientNet-B0 with `weights=None` (no pretraining).
- Replace the classifier head for **binary classification** (1 output neuron).

In [ ]:
baseline_model = models.efficientnet_b0(weights=None)  # TODO: set weights to None (no pretraining)

num_features = baseline_model.classifier[1].in_features
baseline_model.classifier[1] = nn.Linear(num_features, None)  # TODO: output size for binary classification

baseline_model = baseline_model.to(device)

total_params = sum(p.numel() for p in baseline_model.parameters())
print(f'Total parameters: {total_params:,}')

## Task 2: Train the Baseline

Fill in the `None` values:
- **Loss:** `BCEWithLogitsLoss` for binary classification (combines sigmoid + binary cross entropy).
- **Optimizer:** `AdamW` with `lr=1e-3`.

In [ ]:
baseline_criterion = None                                       # TODO: binary cross-entropy loss with logits
baseline_optimizer = optim.AdamW(baseline_model.parameters(), lr=None)  # TODO: learning rate

print('Training baseline (no pretraining) ...')
baseline_history = run_training(
    baseline_model, train_loader, valid_loader,
    baseline_criterion, baseline_optimizer, device, num_epochs=5
)

In [ ]:
# --- GIVEN: Plot baseline results ---
plot_history(baseline_history, 'Baseline (No Pretraining)')

---
# Part 3 -- Strategy 1: Feature Extraction (Freeze Backbone)

Now we load EfficientNet-B0 **with pretrained ImageNet weights** and **freeze the entire backbone**. Only the new classification head is trained. This is the fastest transfer learning approach and works well when your dataset is small and similar to ImageNet.

## Task 3: Load Pretrained Model and Freeze Backbone

Fill in the `None` values:
1. Load EfficientNet-B0 with `pretrained=True`.
2. **Freeze** all backbone parameters by setting `requires_grad` to the correct value.
3. Replace the classifier head for binary classification.

In [ ]:
fe_model = models.efficientnet_b0(pretrained=None)  # TODO: enable pretrained weights

# Freeze ALL backbone parameters
for param in fe_model.parameters():
    param.requires_grad = None  # TODO: prevent gradient updates

# Replace classifier head (newly created layers are trainable by default)
num_features = fe_model.classifier[1].in_features
fe_model.classifier[1] = nn.Linear(num_features, None)  # TODO: output size

fe_model = fe_model.to(device)

## Task 4: Count Trainable Parameters

Fill in the `None` values to count how many parameters are trainable vs total.

In [ ]:
trainable = sum(p.numel() for p in fe_model.parameters() if None)  # TODO: condition for trainable
total     = sum(p.numel() for p in fe_model.parameters())
print(f'Training {trainable:,} / {total:,} parameters ({100*trainable/total:.2f}%)')

## Task 5: Train Only the Head

Fill in the `None` values:
- The optimizer should **only** receive parameters that require gradients.
- Use `filter(lambda p: p.requires_grad, model.parameters())` to select them.

In [ ]:
fe_criterion = nn.BCEWithLogitsLoss()
fe_optimizer = optim.AdamW(
    filter(lambda p: None, fe_model.parameters()),  # TODO: filter condition
    lr=1e-3
)

print('Training with frozen backbone (feature extraction) ...')
fe_history = run_training(
    fe_model, train_loader, valid_loader,
    fe_criterion, fe_optimizer, device, num_epochs=10
)

In [ ]:
# --- GIVEN: Plot feature extraction results ---
plot_history(fe_history, 'Strategy 1: Feature Extraction (Frozen Backbone)')

---
# Part 4 -- Strategy 2: Fine-Tuning Last Layers

Feature extraction is fast but limits how much the model can adapt. **Fine-tuning** unfreezes the deepest backbone layers so they can adjust to your specific domain. We use a **smaller learning rate** for the backbone (to preserve pretrained knowledge) and a **larger one** for the new head.

## Task 6: Freeze Backbone, Then Unfreeze Last Layers

EfficientNet-B0 has 9 feature blocks (`model.features[0]` through `model.features[8]`). We will:
1. Freeze everything.
2. **Unfreeze the last 2 blocks** (`features[7:]`) so they can adapt.
3. Replace the classifier head.

Fill in the `None` values.

In [ ]:
ft_model = models.efficientnet_b0(pretrained=True)

# Step 1: Freeze ALL parameters
for param in ft_model.parameters():
    param.requires_grad = None  # TODO: freeze

# Step 2: Unfreeze the last 2 feature blocks
for param in ft_model.features[None:].parameters():  # TODO: starting index (7 gives last 2 blocks)
    param.requires_grad = None  # TODO: unfreeze

# Step 3: Replace classifier head
num_features = ft_model.classifier[1].in_features
ft_model.classifier[1] = nn.Linear(num_features, 1)

ft_model = ft_model.to(device)

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in ft_model.parameters())
print(f'Training {trainable:,} / {total:,} parameters ({100*trainable/total:.2f}%)')

## Task 7: Differential Learning Rates

Pretrained layers should learn slowly (small LR) to preserve ImageNet features, while the new head can learn faster.

Fill in the `None` values to create **parameter groups** with different learning rates.

In [ ]:
ft_criterion = nn.BCEWithLogitsLoss()

ft_optimizer = optim.AdamW([
    {'params': ft_model.features[7:].parameters(), 'lr': None},  # TODO: small LR for backbone (e.g. 1e-5)
    {'params': ft_model.classifier.parameters(),   'lr': None},  # TODO: larger LR for head (e.g. 1e-3)
])

## Task 8: Train With Fine-Tuning

Run the training loop. The `run_training` helper handles everything -- just pass the right arguments.

In [ ]:
print('Training with fine-tuning (last 2 blocks unfrozen) ...')
ft_history = run_training(
    None, train_loader, valid_loader,  # TODO: which model?
    None, None, device, num_epochs=10  # TODO: criterion and optimizer
)

In [ ]:
# --- GIVEN: Plot fine-tuning results ---
plot_history(ft_history, 'Strategy 2: Fine-Tuning (Last 2 Blocks)')

---
# Part 5 -- Comparison

Let's compare all three approaches side by side.

In [ ]:
# --- GIVEN: Summary comparison ---
strategies = ['Baseline\n(No Pretraining)', 'Feature Extraction\n(Frozen Backbone)', 'Fine-Tuning\n(Last 2 Blocks)']
final_val_accs = [
    baseline_history['val_acc'][-1],
    fe_history['val_acc'][-1],
    ft_history['val_acc'][-1],
]

colors = ['#8c1515', '#0073e6', '#00a676']
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(strategies, final_val_accs, color=colors, width=0.5)

for bar, acc in zip(bars, final_val_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=12)

ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Transfer Learning Strategy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
# Reflection

Answer the following questions in your own words:

1. **Why did the pretrained models outperform the baseline?** What features did they already know?

2. **When would you choose feature extraction over fine-tuning?** Think about dataset size and compute.

3. **Why do we use a smaller learning rate for the backbone layers during fine-tuning?**

4. **What would happen if you had only 50 images per class?** Which strategy would you pick and why?

*Your answers here*